<a name="top"></a>
<img src="../images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 3.1：生成器：参数
**上一篇：[ChiselTest（原 chisel-testers2）](2.6_chiseltest.ipynb)**<br>
**下一篇：[生成器：集合](3.2_collections.ipynb)**

## 动机
对于 Chisel 模块作为代码生成器，必须有某些东西告诉生成器应该如何执行其工作。
在本节中，我们讨论模块参数化、各种方法和 Scala 语言特性。
参数传递实现的丰富度直接关系到所生成电路的丰富度。
参数应提供有用的默认值，易于设置，并防止非法或无意义的取值。
对于更复杂的系统，能够以不会意外影响其他模块使用的方式进行局部覆盖非常有用。

## 环境设置

In [ ]:
val path = System.getProperty("user.dir") + "/../source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 参数传递
Chisel 提供了强大的结构来编写硬件生成器。
生成器是接受一些电路参数并产生电路描述的程序。
在本节中，我们将首先讨论 Chisel 生成器如何获取其参数。

<span style="color:blue">**示例：参数化的 Scala 对象**</span><br>
每个 Chisel `Module` 都是一个 Scala 类，与其他类无异。
回想一下，Scala 类可以像这样参数化：

In [ ]:
class ParameterizedScalaObject(param1: Int, param2: String) {
  println(s"I have parameters: param1 = $param1 and param2 = $param2")
}
val obj1 = new ParameterizedScalaObject(4,     "Hello")
val obj2 = new ParameterizedScalaObject(4 + 2, "World")

<span style="color:blue">**示例：参数化的 Chisel 对象**</span><br>
Chisel 模块可以以相同的方式进行参数化。
以下模块具有用于其所有输入和输出宽度的参数。
运行代码块将打印生成的 Verilog。
尝试调整参数，并检查输出是否反映了新的参数。

In [ ]:
class ParameterizedWidthAdder(in0Width: Int, in1Width: Int, sumWidth: Int) extends Module {
  require(in0Width >= 0)
  require(in1Width >= 0)
  require(sumWidth >= 0)
  val io = IO(new Bundle {
    val in0 = Input(UInt(in0Width.W))
    val in1 = Input(UInt(in1Width.W))
    val sum = Output(UInt(sumWidth.W))
  })
  // a +& b includes the carry, a + b does not
  io.sum := io.in0 +& io.in1
}

println(getVerilog(new ParameterizedWidthAdder(1, 4, 6)))

上面的代码块中有一些 `require(...)` 语句。
这些是预展开断言，当你的生成器仅适用于某些参数化，或者某些参数化是互斥的或无意义时，这些断言非常有用。
上面的代码块检查宽度是否为非负数。

还有一个用于仿真时断言的独立结构，称为 `assert(...)`。

## 使用参数化模块进行排序
以下代码块是一个参数化排序器，类似于 2.3 模块中的 `Sort4`。
与之前具有参数化宽度 IO 的加法器示例不同，此示例具有固定的 IO。
参数控制模块内部生成什么硬件。
![Sort4](images/Sorter4.png)
<span style="color:blue">**示例：参数化的 4 输入排序器**</span><br>
与 2.3 不同，此实现被参数化为降序或升序排序。

In [ ]:
/** Sort4 sorts its 4 inputs to its 4 outputs */
class Sort4(ascending: Boolean) extends Module {
  val io = IO(new Bundle {
    val in0 = Input(UInt(16.W))
    val in1 = Input(UInt(16.W))
    val in2 = Input(UInt(16.W))
    val in3 = Input(UInt(16.W))
    val out0 = Output(UInt(16.W))
    val out1 = Output(UInt(16.W))
    val out2 = Output(UInt(16.W))
    val out3 = Output(UInt(16.W))
  })
    
  // this comparison funtion decides < or > based on the module's parameterization
  def comp(l: UInt, r: UInt): Bool = {
      if (ascending) {
        l < r
      } else {
        l > r
    }
  }

  val row10 = Wire(UInt(16.W))
  val row11 = Wire(UInt(16.W))
  val row12 = Wire(UInt(16.W))
  val row13 = Wire(UInt(16.W))

  when(comp(io.in0, io.in1)) {
    row10 := io.in0            // preserve first two elements
    row11 := io.in1
  }.otherwise {
    row10 := io.in1            // swap first two elements
    row11 := io.in0
  }

  when(comp(io.in2, io.in3)) {
    row12 := io.in2            // preserve last two elements
    row13 := io.in3
  }.otherwise {
    row12 := io.in3            // swap last two elements
    row13 := io.in2
  }

  val row21 = Wire(UInt(16.W))
  val row22 = Wire(UInt(16.W))

  when(comp(row11, row12)) {
    row21 := row11            // preserve middle 2 elements
    row22 := row12
  }.otherwise {
    row21 := row12            // swap middle two elements
    row22 := row11
  }

  val row20 = Wire(UInt(16.W))
  val row23 = Wire(UInt(16.W))
  when(comp(row10, row13)) {
    row20 := row10            // preserve the first and the forth elements
    row23 := row13
  }.otherwise {
    row20 := row13            // swap the first and the forth elements
    row23 := row10
  }

  when(comp(row20, row21)) {
    io.out0 := row20            // preserve first two elements
    io.out1 := row21
  }.otherwise {
    io.out0 := row21            // swap first two elements
    io.out1 := row20
  }

  when(comp(row22, row23)) {
    io.out2 := row22            // preserve first two elements
    io.out3 := row23
  }.otherwise {
    io.out2 := row23            // swap first two elements
    io.out3 := row22
  }
}



// Here are the testers
test(new Sort4(true)) { c => 
  c.io.in0.poke(3.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(9.U)
  c.io.in3.poke(12.U)
  c.io.out0.expect(3.U)
  c.io.out1.expect(6.U)
  c.io.out2.expect(9.U)
  c.io.out3.expect(12.U)

  c.io.in0.poke(13.U)
  c.io.in1.poke(4.U)
  c.io.in2.poke(6.U)
  c.io.in3.poke(1.U)
  c.io.out0.expect(1.U)
  c.io.out1.expect(4.U)
  c.io.out2.expect(6.U)
  c.io.out3.expect(13.U)

  c.io.in0.poke(13.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(4.U)
  c.io.in3.poke(1.U)
  c.io.out0.expect(1.U)
  c.io.out1.expect(4.U)
  c.io.out2.expect(6.U)
  c.io.out3.expect(13.U)
}
test(new Sort4(false)) { c =>
  c.io.in0.poke(3.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(9.U)
  c.io.in3.poke(12.U)
  c.io.out0.expect(12.U)
  c.io.out1.expect(9.U)
  c.io.out2.expect(6.U)
  c.io.out3.expect(3.U)

  c.io.in0.poke(13.U)
  c.io.in1.poke(4.U)
  c.io.in2.poke(6.U)
  c.io.in3.poke(1.U)
  c.io.out0.expect(13.U)
  c.io.out1.expect(6.U)
  c.io.out2.expect(4.U)
  c.io.out3.expect(1.U)

  c.io.in0.poke(1.U)
  c.io.in1.poke(6.U)
  c.io.in2.poke(4.U)
  c.io.in3.poke(13.U)
  c.io.out0.expect(13.U)
  c.io.out1.expect(6.U)
  c.io.out2.expect(4.U)
  c.io.out3.expect(1.U)
}
println("SUCCESS!!") // Scala Code: if we get here, our tests passed!

---
# Option 和默认参数

有时函数有时会返回一个值，有时则不返回。Scala 不是在无法返回值时直接报错，而是有一种机制可以将其编码到类型系统中。

<span style="color:blue">**示例：错误的 Map 索引调用**</span><br>
在下面的示例中，我们有一个包含几组键/值对的 map。如果我们尝试访问一个缺失的键/值对，则会得到一个运行时错误：

In [ ]:
val map = Map("a" -> 1)
val a = map("a")
println(a)
val b = map("b")
println(b)

<span style="color:blue">**示例：获取不确定的索引**</span><br>
然而，`Map` 提供了另一种访问键值的方法，即通过 **get** 方法。使用此方法会返回一个抽象类 `Option` 的值。`Option` 有两个子类，`Some` 和 `None`。

In [ ]:
val map = Map("a" -> 1)
val a = map.get("a")
println(a)
val b = map.get("b")
println(b)

正如你将在后面的章节中看到的，`Option` 极其重要，因为它允许用户使用 match 语句来检查 Scala 类型和值。

<span style="color:blue">**示例：Get Or Else！**</span><br>
像 `Map` 一样，`Option` 也有一个 `get` 方法，如果在 `None` 上调用会报错。对于这些情况，我们可以使用 **`getOrElse`** 提供一个默认值。

In [ ]:
val some = Some(1)
val none = None
println(some.get)          // Returns 1
// println(none.get)       // Errors!
println(some.getOrElse(2)) // Returns 1
println(none.getOrElse(2)) // Returns 2

## 带默认值的参数的 Option

当对象或函数有大量参数时，每次都完整指定它们可能会很繁琐且容易出错。
在 module 1 中，你已经了解了命名参数和参数默认值。
有时，参数没有一个好的默认值。
在这些情况下，`Option` 可以与默认值 `None` 一起使用。

<span style="color:blue">**示例：可选的复位值**</span><br>
以下展示了一个将其输入延迟一个时钟周期的模块。
如果 `resetValue = None`（这是默认值），寄存器将没有复位值，并被初始化为垃圾值。
这避免了使用超出正常范围的值来表示"无"的常见但丑陋的情况，例如使用 -1 作为复位值来指示该寄存器不复位。

In [ ]:
class DelayBy1(resetValue: Option[UInt] = None) extends Module {
    val io = IO(new Bundle {
        val in  = Input( UInt(16.W))
        val out = Output(UInt(16.W))
    })
    val reg = if (resetValue.isDefined) { // resetValue = Some(number)
        RegInit(resetValue.get)
    } else { //resetValue = None
        Reg(UInt())
    }
    reg := io.in
    io.out := reg
}

println(getVerilog(new DelayBy1))
println(getVerilog(new DelayBy1(Some(3.U))))

---
# Match/Case 语句
Scala 的*匹配*概念贯穿于整个 Chisel，需要成为任何 Chisel 程序员基本理解的一部分。Scala 提供了 match 运算符，支持：
- 简单的替代测试，类似于 C 语言的 *switch* 语句
- 更复杂的值的临时组合测试
- 根据变量的类型（当其类型未知或未充分指定时）采取行动，例如当
  - 变量来自异构列表时 ```val mixedList = List(1, "string", false)```
  - 或者变量已知是某个超类的成员，但不知道是哪个具体的子类时
- 提取字符串中由*正则表达式*指定的子字符串


<span style="color:blue">**示例：值匹配**</span><br>
以下示例，根据我们 **match** 的变量的**值**，执行不同的 **case** 语句：

In [ ]:
// y is an integer variable defined somewhere else in the code
val y = 7
/// ...
val x = y match {
  case 0 => "zero" // One common syntax, preferred if fits in one line
  case 1 =>        // Another common syntax, preferred if does not fit in one line.
      "one"        // Note the code block continues until the next case
  case 2 => {      // Another syntax, but curly braces are not required
      "two"
  }
  case _ => "many" // _ is a wildcard that matches all values
}
println("y is " + x)

match 运算符检查可能的值，并为每个 case 返回一个字符串。需要注意的几点：
- 每个 `=>` 运算符后面的代码块会一直执行，直到到达 match 的结束大括号或下一个 case 语句。
- 匹配按照 case 语句的顺序进行搜索，一旦某个 case 语句被匹配，就不再对其他 case 语句进行检查。
- 使用下划线作为通配符，用于处理未找到的任何值。

<span style="color:blue">**示例：多值匹配**</span><br>
此外，可以同时匹配多个变量。这里是一个使用 match 语句和值元组实现真值表的简单示例：

In [ ]:
def animalType(biggerThanBreadBox: Boolean, meanAsCanBe: Boolean): String = {
  (biggerThanBreadBox, meanAsCanBe) match {
    case (true, true) => "wolverine"
    case (true, false) => "elephant"
    case (false, true) => "shrew"
    case (false, false) => "puppy"
  }
}
println(animalType(true, true))

<span style="color:blue">**示例：类型匹配**</span><br>
Scala 是一种强类型语言，因此所有对象的类型在运行时都是已知的。我们可以使用 **match 语句** 来利用此类型信息控制流程：

In [ ]:
val sequence = Seq("a", 1, 0.0)
sequence.foreach { x =>
  x match {
    case s: String => println(s"$x is a String")
    case s: Int    => println(s"$x is an Int")
    case s: Double => println(s"$x is a Double")
    case _ => println(s"$x is an unknown type!")
  }
}

<span style="color:blue">**示例：多类型匹配**</span><br>
如果你想匹配一个值是否具有多种类型之一，请使用以下语法。*注意，匹配时必须使用 `_`。*

In [ ]:
val sequence = Seq("a", 1, 0.0)
sequence.foreach { x =>
  x match {
    case _: Int | _: Double => println(s"$x is a number!")
    case _ => println(s"$x is an unknown type!")
  }
}

<span style="color:blue">**示例：类型匹配和擦除**</span><br>
类型匹配有一些限制。因为 Scala 运行在 JVM 上，而 JVM 不维护多态类型，所以无法在运行时对它们进行匹配（因为它们都被擦除了）。请注意，以下示例总是匹配第一个 case 语句，因为 `[String]`、`[Int]` 和 `[Double]` 多态类型被擦除了，而 case 语句实际上只是匹配一个 `Seq`。

In [ ]:
val sequence = Seq(Seq("a"), Seq(1), Seq(0.0))
sequence.foreach { x =>
  x match {
    case s: Seq[String] => println(s"$x is a String")
    case s: Seq[Int]    => println(s"$x is an Int")
    case s: Seq[Double] => println(s"$x is a Double")
  }
}

请注意，Scala 编译器通常会在你实现类似上述示例的代码时给出警告。

<span style="color:blue">**示例：可选的复位值匹配**</span><br>
以下代码块展示了使用 match 结构而非 `if/else` 的相同 `DelayBy1` 模块。

In [ ]:
class DelayBy1(resetValue: Option[UInt] = None) extends Module {
  val io = IO(new Bundle {
    val in  = Input( UInt(16.W))
    val out = Output(UInt(16.W))
  })
  val reg = resetValue match {
    case Some(r) => RegInit(r)
    case None    => Reg(UInt())
  }
  reg := io.in
  io.out := reg
}

println(getVerilog(new DelayBy1))
println(getVerilog(new DelayBy1(Some(3.U))))

---
# 具有可选字段的 IO

有时我们希望 IO 可以被选择性地包含或排除。
也许有一些内部状态在调试时方便查看，但你想在生成器用于系统时将其隐藏。
也许你的生成器有一些输入，在每种情况下都不需要连接，因为有合理的默认值。

<span style="color:blue">**示例：使用 Option 的可选 IO**</span><br>
可选的 Bundle 字段是实现此功能的一种方法。
在下面的示例中，我们展示了一个可选择性地接受进位输入的一位加法器。
如果包含了进位，`io.carryIn` 将具有类型 `Some[UInt]` 并被包含在 IO Bundle 中。
如果不包含进位，`io.carryIn` 将具有类型 `None` 并从 IO Bundle 中排除。

In [ ]:
class HalfFullAdder(val hasCarry: Boolean) extends Module {
  val io = IO(new Bundle {
    val a = Input(UInt(1.W))
    val b = Input(UInt(1.W))
    val carryIn = if (hasCarry) Some(Input(UInt(1.W))) else None
    val s = Output(UInt(1.W))
    val carryOut = Output(UInt(1.W))
  })
  val sum = io.a +& io.b +& io.carryIn.getOrElse(0.U)
  io.s := sum(0)
  io.carryOut := sum(1)
}

test(new HalfFullAdder(false)) { c =>
  require(!c.hasCarry, "DUT must be half adder")
  // 0 + 0 = 0
  c.io.a.poke(0.U)
  c.io.b.poke(0.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(0.U)
  // 0 + 1 = 1
  c.io.b.poke(1.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
  // 1 + 1 = 2
  c.io.a.poke(1.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
  // 1 + 0 = 1
  c.io.b.poke(0.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
}

test(new HalfFullAdder(true)) { c =>
  require(c.hasCarry, "DUT must be half adder")
  c.io.carryIn.get.poke(0.U)
  // 0 + 0 + 0 = 0
  c.io.a.poke(0.U)
  c.io.b.poke(0.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(0.U)
  // 0 + 0 + 1 = 1
  c.io.b.poke(1.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
  // 0 + 1 + 1 = 2
  c.io.a.poke(1.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
  // 0 + 1 + 0 = 1
  c.io.b.poke(0.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)

  c.io.carryIn.get.poke(1.U)
  // 1 + 0 + 0 = 1
  c.io.a.poke(0.U)
  c.io.b.poke(0.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(0.U)
  // 1 + 0 + 1 = 2
  c.io.b.poke(1.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
  // 1 + 1 + 1 = 3
  c.io.a.poke(1.U)
  c.io.s.expect(1.U)
  c.io.carryOut.expect(1.U)
  // 1 + 1 + 0 = 2
  c.io.b.poke(0.U)
  c.io.s.expect(0.U)
  c.io.carryOut.expect(1.U)
}

println("SUCCESS!!") // Scala Code: if we get here, our tests passed!

<span style="color:blue">**示例：使用零宽度线网的可选 IO**</span><br>
实现与 `Option` 类似功能的另一种方法是使用零宽度线网。
Chisel 类型允许具有零宽度。
宽度为零的 IO 会从生成的 Verilog 中被修剪掉，任何试图使用零宽度线网值的东西都会得到常量零。
如果零是一个合理的默认值，那么零宽度线网会很方便，因为它们省去了在 option 上进行匹配或调用 `getOrElse` 的需要。

In [ ]:
class HalfFullAdder(val hasCarry: Boolean) extends Module {
  val io = IO(new Bundle {
    val a = Input(UInt(1.W))
    val b = Input(UInt(1.W))
    val carryIn = Input(if (hasCarry) UInt(1.W) else UInt(0.W))
    val s = Output(UInt(1.W))
    val carryOut = Output(UInt(1.W))
  })
  val sum = io.a +& io.b +& io.carryIn
  io.s := sum(0)
  io.carryOut := sum(1)
}
println("Half Adder:")
println(getVerilog(new HalfFullAdder(false)))
println("\n\nFull Adder:")
println(getVerilog(new HalfFullAdder(true)))

---
# Implicits（隐式）
在编程时，经常会有需要大量样板代码的情况。为了处理这种情况，Scala 引入了**implicits**（隐式）的概念，它允许编译器为你做一些语法糖。因为有很多事情发生在幕后，implicits 看起来可能很神奇。本节通过一些基本示例来介绍它们是什么以及它们通常在哪里使用。

## 隐式参数
有时，你的代码需要在一系列函数调用的深处访问某种顶层变量。与其手动将这个变量传递给每个函数调用，你可以使用隐式参数来为你完成这项工作。

<span style="color:blue">**示例：隐式的猫**</span><br>
在下面的示例中，我们可以隐式或显式地传递猫的数量。

In [ ]:
object CatDog {
  implicit val numberOfCats: Int = 3
  //implicit val numberOfDogs: Int = 5

  def tooManyCats(nDogs: Int)(implicit nCats: Int): Boolean = nCats > nDogs
    
  val imp = tooManyCats(2)    // Argument passed implicitly!
  val exp = tooManyCats(2)(1) // Argument passed explicitly!
}
CatDog.imp
CatDog.exp

这里发生了什么？首先，我们定义一个隐式值 **numberOfCats**。在给定的作用域中，**对于给定类型，只能有一个隐式值**。然后，我们定义一个函数，它接受两个参数列表；第一个是任何显式参数，第二个是任何隐式参数。当我们调用 **tooManyCats** 时，我们可以省略第二个隐式参数列表（让编译器为我们找到它），或者显式提供一个参数（可以与隐式值不同）。

以下情况下隐式参数可能会*失败*：
- 在作用域中定义了同一类型的两个或多个隐式值
- 如果编译器无法找到函数调用所需的隐式值

<span style="color:blue">**示例：隐式日志记录**</span><br>
下一个代码块展示了如何使用隐式参数在 Chisel 生成器中实现日志记录。

***注意：在 Scala 中有更好的日志记录方法！***

In [ ]:
sealed trait Verbosity
implicit case object Silent extends Verbosity
case object Verbose extends Verbosity

class ParameterizedWidthAdder(in0Width: Int, in1Width: Int, sumWidth: Int)(implicit verbosity: Verbosity)
extends Module {
  def log(msg: => String): Unit = verbosity match {
    case Silent =>
    case Verbose => println(msg)
  }
  require(in0Width >= 0)
  log(s"in0Width of $in0Width OK")
  require(in1Width >= 0)
  log(s"in1Width of $in1Width OK")
  require(sumWidth >= 0)
  log(s"sumWidth of $sumWidth OK")
  val io = IO(new Bundle {
    val in0 = Input(UInt(in0Width.W))
    val in1 = Input(UInt(in1Width.W))
    val sum = Output(UInt(sumWidth.W))
  })
  log("Made IO")
  io.sum := io.in0 + io.in1
  log("Assigned output")
}

println(getVerilog(new ParameterizedWidthAdder(1, 4, 5)))
println(getVerilog(new ParameterizedWidthAdder(1, 4, 5)(Verbose)))

## 隐式转换
与隐式参数一样，隐式函数（也称为**隐式转换**）用于减少样板代码。更具体地说，它们用于自动将一个 Scala 对象转换为另一个。

<span style="color:blue">**示例：隐式转换**</span><br>
在下面的示例中，我们有两个类，`Animal` 和 `Human`。`Animal` 有 `species` 字段，而 `Human` 没有。然而，通过实现一个隐式转换，我们可以在 `Human` 上调用 `species`。

In [ ]:
class Animal(val name: String, val species: String)
class Human(val name: String)
implicit def human2animal(h: Human): Animal = new Animal(h.name, "Homo sapiens")
val me = new Human("Adam")
println(me.species)

通常，implicits 可能会使你的代码令人困惑，因此我们建议你将其作为最后的手段。首先尝试继承、traits 或方法重载。

---
# 生成器示例
以下示例展示了一个 1 位输入 Mealy 机的生成器。
它有一个基于[维基百科](https://en.wikipedia.org/wiki/Mealy_machine#/media/File:Mealy.png)示例的测试。
阅读代码并尝试理解其运作方式。

<span style="color:blue">**示例：Mealy 机**</span><br>
尝试在下面的代码块中创建你自己的 Mealy 机生成器参数化并编写你自己的测试。

In [ ]:
// Mealy machine has
case class BinaryMealyParams(
  // number of states
  nStates: Int,
  // initial state
  s0: Int,
  // function describing state transition
  stateTransition: (Int, Boolean) => Int,
  // function describing output
  output: (Int, Boolean) => Int
) {
  require(nStates >= 0)
  require(s0 < nStates && s0 >= 0)
}

class BinaryMealy(val mp: BinaryMealyParams) extends Module {
  val io = IO(new Bundle {
    val in = Input(Bool())
    val out = Output(UInt())
  })

  val state = RegInit(UInt(), mp.s0.U)

  // output zero if no states
  io.out := 0.U
  for (i <- 0 until mp.nStates) {
    when (state === i.U) {
      when (io.in) {
        state  := mp.stateTransition(i, true).U
        io.out := mp.output(i, true).U
      }.otherwise {
        state  := mp.stateTransition(i, false).U
        io.out := mp.output(i, false).U
      }
    }
  }
}

// example from https://en.wikipedia.org/wiki/Mealy_machine
val nStates = 3
val s0 = 2
def stateTransition(state: Int, in: Boolean): Int = {
  if (in) {
    1
  } else {
    0
  }
}
def output(state: Int, in: Boolean): Int = {
  if (state == 2) {
    return 0
  }
  if ((state == 1 && !in) || (state == 0 && in)) {
    return 1
  } else {
    return 0
  }
}

val testParams = BinaryMealyParams(nStates, s0, stateTransition, output)

test(new BinaryMealy(testParams)) { c =>
  c.io.in.poke(false.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(0.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(false.B)
  c.io.out.expect(1.U)
  c.clock.step(1)
  c.io.in.poke(true.B)
  c.io.out.expect(1.U)
}

println("SUCCESS!!") // Scala Code: if we get here, our tests passed!

---
# 你已完成！

[返回顶部。](#top)